# Popularity bias in recommenders

Cleans Amazon Reviews 2023 data, splits it by time, then trains and evaluates four recommenders
(`mostpop`, `itemcf`, `mf`, `gru`) on accuracy and popularity-bias metrics.

Run this notebook from the project folder. Data: https://amazon-reviews-2023.github.io/

In [ ]:
%pip install -q -r requirements.txt

## Settings

In [ ]:
CATEGORY = "Clothing_Shoes_and_Jewelry"
RAW_REVIEWS = "Clothing_Shoes_and_Jewelry.jsonl"
K_CORE = 2
MODELS = ["mostpop", "itemcf", "mf", "gru"]
SAMPLE_EVERY = None   # e.g. 50 keeps 1 in 50 users for a quick run

## Clean the raw reviews
Skip this if `data/<CATEGORY>_interactions.parquet` already exists.

In [ ]:
!python prepare.py {RAW_REVIEWS} {CATEGORY}

## Optional: sample users

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

if SAMPLE_EVERY:
    parts = []
    for batch in pq.ParquetFile(f"data/{CATEGORY}_interactions.parquet").iter_batches(batch_size=500_000):
        df = batch.to_pandas()
        parts.append(df[pd.util.hash_pandas_object(df.user_id, index=False).to_numpy() % SAMPLE_EVERY == 0])
    CATEGORY = f"{CATEGORY}_sample"
    pd.concat(parts, ignore_index=True).to_parquet(f"data/{CATEGORY}_interactions.parquet", index=False)

## Split into train / validation / test

In [ ]:
!python split.py {CATEGORY} {K_CORE}

## Train and evaluate

In [ ]:
for m in MODELS:
    !python run.py {CATEGORY} {K_CORE} {m}

## Results

In [ ]:
results = pd.read_csv("results.csv")
results[results.category == CATEGORY]